# Demo of Framework execution limited to the hazard determinant

This notebook shows the steps of the Framework related to the hazard determinant.
Given a system to analyse, the outcome is a mathematical object which contains information on the drivers within the hazard determinant affecting the system. Ensembles of these mathematical objects are then created varying indicators of the drivers and climate data used to evaluate them.

The content and the Python code are general: they are valid for any choice of drivers, their indicators and data.
To show in practice how to execute this part of the Framework, a fictional risk assessment is performed. The example is described through text in *italic* and blocks of code introduced by the comment `# Example`.

## Step 1

Given the system to analyse, select drivers within the hazard determinant which could affect the system.

Note that at this point no reference to the componenents of the system, i.e. the assets, is made.
In general this distinction is not necessary for the part of the Framework regarding the hazard determinant. However, thinking to the system in terms of its components could help the analyst to identify the drivers affecting the system in its entirety.

*The system is Rome Ciampino airport. It is found to be affected only by drivers "heat wave" and "heavy precipitation". They are chosen for the next steps of the analysis.*

In [ ]:
# Example: insert drivers in Python code.

drivers_kw = ["heat_wave", "heavy_precipitation"]

## Step 2

Choose the indicators suitable to describe each driver.

*Indicators already provided by the xclim Python package are chosen for ease. [Their list](https://xclim.readthedocs.io/en/stable/indicators.html) is available in xclim official documentation, together with [a guide](https://xclim.readthedocs.io/en/stable/notebooks/extendxclim.html#Defining-new-indicators) on how to build new indicators.*  
*The chosen xclim indicators of heat wave are: [heat_wave_frequency](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_frequency), [heat_wave_max_length](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_max_length), [heat_wave_total_length](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.heat_wave_total_length) and [tx_max](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.tx_max).*  
*The chosen xclim indicators of heavy precipitation are: [max_n_day_precipitation_amount](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.max_n_day_precipitation_amount), [wetdays](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.wetdays) and [cwd](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.cwd). Indicator max_n_day_precipitation_amount is considered with two different window sizes.*

## Step 3

Implement the indicators in Python language.

One option is to load indicators as Python objects from a YAML file. Further details are available in [the specific section](https://xclim.readthedocs.io/en/stable/api.html#yaml-file-structure) of xclim official documentation.

In [ ]:
import pathlib

import xclim

In [ ]:
indicators = {}
for driver_kw in drivers_kw:
    _module = xclim.core.indicator.build_indicator_module_from_yaml(pathlib.Path(f"{driver_kw}.yml"))
    indicators.update(dict(_module.iter_indicators()))

## Step 4

Select climate data. Choose spatial boundaries to describe the local climate surrounding the system with sufficient accuracy and select the baseline period.
The selected data must contain the climate variables needed to compute the indicators.

Some preprocessing may be performed on data once loaded in Python. For instance, different datasets may refer to the same climate variable with different names and they need to be renamed to the same string to make the following Python code work with no user supervision.
For variables names [the xclim conventions](https://xclim.readthedocs.io/en/stable/readme.html#conventions) are followed. In particular, the available climate variables available natively in xclim are stored in `xclim.core.utils.VARIABLES`.

*Climate data used in the example are from ERA5Land by ECMWF and from ERA5@2km by CMCC. The selected baseline period is 1980-2010. Two different datasets serve as example for the creation of an ensemble by varying climate data.*
*The selected indicators need variables "tasmin", "tasmax" and "pr" as inputs. First two variables are derived from variable "tas", while variable "pr" is derived from the precipitation amount, which is represented here with variable "tp". Hence data for climate variables "tas" and "tp" are sufficient for the following calculations.*

In [ ]:
import xarray

In [ ]:
# Example: choose datasets. To improve data management, each dataset is stored
# in a folder with same name and split in a folder for each climate variable.

datasets_kw = ["era5-land", "era5-2km"]
variables_kw = ["2m_temperature", "total_precipitation"]
baseline_period = slice("1980", "2010")

In [ ]:
ds_hourly = {}
for dataset_kw in datasets_kw:
    # Load different variables of the same dataset in the same xarray dataset.
    ds_hourly[dataset_kw] = xarray.Dataset()
    for variable_kw in variables_kw:
        path_data = pathlib.Path("data", dataset_kw, variable_kw)
        _dataset = xarray.open_mfdataset(path_data.iterdir())
        ds_hourly[dataset_kw] = xarray.merge(
            [ds_hourly[dataset_kw], _dataset],
            combine_attrs="drop_conflicts"
        )
    # Restrict data to the selected baseline period.
    ds_hourly[dataset_kw] = ds_hourly[dataset_kw].sel(time=baseline_period, drop=True)

In [ ]:
# Example: set same names to same climate variables in every dataset.

ds_hourly["era5-land"] = ds_hourly["era5-land"].rename({
    "t2m": "tas"
})
ds_hourly["era5-2km"] = ds_hourly["era5-2km"].rename({
    "T_2M": "tas",
    "TOT_PREC": "tp"
})

In [ ]:
# Example: set CF Standard Name where missing, needed for automatic conversions.

ds_hourly["era5-land"]["tp"].attrs["standard_name"] = "lwe_thickness_of_precipitation_amount"

In [ ]:
# Example: evaluate derived climate variables needed as inputs of indicators.

# Use new xarray datasets when introducing a new sampling frequency
# to have consistent time units.
ds_daily = {}
for dataset_kw in datasets_kw:
    ds_daily[dataset_kw] = xarray.Dataset()
    # Evaluate tasmin and tasmax.
    _resample = ds_hourly[dataset_kw]["tas"].resample(time="1D")
    ds_daily[dataset_kw]["tasmin"] = _resample.min()
    ds_daily[dataset_kw]["tasmax"] = _resample.max()
    # Resample tp at the sampling frequency needed to obtain pr.
    _resample = ds_hourly[dataset_kw]["tp"].resample(time="1D")
    ds_daily[dataset_kw]["tp"] = _resample.mean()

In [ ]:
# Example: evaluate pr, this conversion can not be done automatically.

for dataset_kw in datasets_kw:
    ds_daily[dataset_kw]["pr"] = xclim.core.units.convert_units_to(
        source=ds_daily[dataset_kw]["tp"],
        target="mmday",
        context="hydro"
    )

## Step 5

Evaluate indicators on each dataset.
The outcome is an ensemble of mathematical objects, each being a collection of yearly averages of the evaluated indicators. There is a member of the ensemble for each dataset.

*Only two datasets are used to build the ensemble, hence resulting statistical analysis is not accurate. The purpose of this example is limited to show how the Framework works, not to draw effective conclusions on its application.*

In [ ]:
# Indicators are evaluated separately for each dataset
# because data could be associated to different geographic coordinate systems.
# If this problem does not apply, then all datasets can be stored in the same
# xarray Dataset object with an additional dimension identifying each of them.
ds_member = {}
for dataset_kw in datasets_kw:
    ds_member[dataset_kw] = xarray.Dataset()
    for indicator_kw, indicator in indicators.items():
        ds_member[dataset_kw][indicator_kw] = indicator(ds=ds_daily[dataset_kw])

## Step 6

Create ensemble by varying the parameters of selected indicators.

With this procedure, an ensemble for each dataset is created.
Note that, given a dataset, the outcome of step 5 is only one member of the ensemble that is created in this step.

From an implementation point of view, in this step an xarray Dataset object is created for each indicator. This choice is made to avoid sparse matrices, an issue related to how xarray Dataset objects are structured: in general each indicator has different parameters and each parameter can assume different values (e.g. two indicators both rely on parameter `thresh_tasmin`, but the number of values chosen as arguments for each of them is different), hence if all combinations of these values were inserted in a single xarray Dataset object, all combined as coordinates in the dimension of the corresponding parameter, the resulting object would contain a huge number of empty cells (i.e. NaN values), wasting computing resources needlessly.
The empty cells are positioned at coordinates which are not valid arguments for the indicators (e.g. the xclim [`tx_max`](https://xclim.readthedocs.io/en/stable/api_indicators.html#xclim.indicators.atmos.tx_max) indicator expects parameter `tasmax` but not parameter `thresh_tasmax`, hence any value of the latter would be an invalid coordinate for the outcomes of this indicator).

In this step, more in detail:
1. Identify the parameters which can be varied for each indicator. 
2. For each parameter decide a set of values that it can take. These sets can be intervals, which ideally are continuous, hence to allow computations on them they are first discretised into sequences of values, separated by a given resolution, with the endpoints included. In the implementation, the number of values in the interval is requested instead of the resolution for increased precision.
3. Evaluate each indicator for every combination of its arguments (i.e. the values of its parameters). The outcomes are inserted in an xarray Dataset object for each indicator, with the varied parameters as additional dimensions and the parameter values as coordinates.

*Same values are used for same parameters in different indicators (e.g. use same sequence of values for `thresh_tasmin` and `thresh_tasmax` parameters in every indicator that uses them). The same number of values is considered to check easily the number of operations. Moreover to speed up the computations, they are performed only for one indicator, hence a single xarray Dataset object results from this step.*

In [ ]:
from framework.packages.xclim import get_variable_parameters

In [ ]:
parameters = {}
# Prepare the storage for the xclim.DataArray objects
# which will be the additional coordinates.
for indicator_kw, indicator in indicators.items():
    parameters[indicator_kw] = get_variable_parameters(indicator)

In [ ]:
from framework.ensemble.intervals import IntervalQuantity, IntervalInteger

In [ ]:
# Example: same values are used for same parameters in different indicators.

intervals = {}

interval_kw = "thresh_tasmin"
intervals[interval_kw] = IntervalQuantity(
    left=xclim.units.units.Quantity("13.0 degC"),
    right=xclim.units.units.Quantity("45.0 degC"),
    name=interval_kw
)

interval_kw = "thresh_tasmax"
intervals[interval_kw] = IntervalQuantity(
    left=xclim.units.units.Quantity("3.0 degC"),
    right=xclim.units.units.Quantity("27.0 degC"),
    name=interval_kw
)

interval_kw = "window"
intervals[interval_kw] = IntervalInteger(
    left=2,
    right=4,
    name=interval_kw
)

interval_kw = "thresh"
intervals[interval_kw] = IntervalQuantity(
    left=xclim.units.units.Quantity("0.3 mmday"),
    right=xclim.units.units.Quantity("40.0 mmday"),
    name=interval_kw
)

# Discretise the intervals using the same number of values for every parameter.
for indicator_kw in indicators.keys():
    for parameter_kw in parameters[indicator_kw].keys():
        interval = intervals[parameter_kw]
        parameters[indicator_kw][parameter_kw] = interval.to_dataarray(
            num=3,
            name=interval.name,
            dims=interval.name
        )

In [ ]:
from framework.packages.xclim import compute_variations

In [ ]:
# Example: computations are perfomed for every indicator.

ds_hazard = {}
# Calculations are performed independently for each dataset to address
# the issue of different geographic coordinate systems as in step 5.
for dataset_kw in datasets_kw:
    ds_hazard[dataset_kw] = {}
    for indicator_kw, indicator in indicators.items():
        ds_hazard[dataset_kw][indicator_kw] = compute_variations(
            indicator=indicator,
            variations=parameters[indicator_kw],
            ds=ds_daily[dataset_kw]
        )

In [ ]:
# Close datasets to release memory.
for dataset_kw in datasets_kw:
    ds_hourly[dataset_kw].close()
    ds_daily[dataset_kw].close()
    ds_member[dataset_kw].close()
    for ds in ds_hazard[dataset_kw].values():
        ds.close()